In [1]:
import pandas as pd
import numpy as np

# Synthetic messy dataset generator
np.random.seed(42)
n = 100

data = {
    'Customer_ID': [101 + i for i in range(n-5)] + [101, 102, 103, 104, 105], # Duplicate IDs
    'Join_Date': np.random.choice(['2023-01-15', '15/01/2023', 'Jan 15, 2023', np.nan, '2023/05/20'], n),
    'Gender': np.random.choice(['Male', 'male', 'M', 'Female', 'female', 'F', np.nan], n),
    'Age': np.random.choice([22, 35, 150, -5, 28, np.nan, 45, 30], n), # Anomalies: 150, -5
    'Income': np.random.choice(['$45,000', '50000', '65,000.50', 'UNKNOWN', np.nan, '$120,000'], n)
}

df_messy = pd.DataFrame(data)
df_messy.to_csv('messy_dataset.csv', index=False)
print("Messy dataset saved as 'messy_dataset.csv'.")

Messy dataset saved as 'messy_dataset.csv'.


In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('messy_dataset.csv')

# Record initial baseline state
before_rows = len(df)
before_nulls = df.isnull().sum().sum()
before_duplicates = df.duplicated().sum()
before_dtypes = df.dtypes.to_dict()

print("--- INITIAL DATA QUALITY REPORT ---")
print(f"Total Rows: {before_rows}")
print(f"Duplicate Rows: {before_duplicates}")
print("\nMissing Values per Column:")
print(df.isnull().sum())
print("\nColumn Data Types:")
print(df.dtypes)
print("\nUnique Values Sample:")
print(df.head(10))

--- INITIAL DATA QUALITY REPORT ---
Total Rows: 100
Duplicate Rows: 0

Missing Values per Column:
Customer_ID     0
Join_Date      26
Gender         19
Age            11
Income         15
dtype: int64

Column Data Types:
Customer_ID      int64
Join_Date          str
Gender             str
Age            float64
Income             str
dtype: object

Unique Values Sample:
   Customer_ID     Join_Date  Gender    Age    Income
0          101           NaN  Female  150.0   UNKNOWN
1          102    2023/05/20     NaN    NaN     50000
2          103  Jan 15, 2023    Male   30.0   $45,000
3          104    2023/05/20  Female   30.0   UNKNOWN
4          105    2023/05/20    male   30.0   UNKNOWN
5          106    15/01/2023    Male   22.0   $45,000
6          107  Jan 15, 2023     NaN   -5.0     50000
7          108  Jan 15, 2023     NaN   22.0   $45,000
8          109  Jan 15, 2023       F    NaN  $120,000
9          110    2023/05/20  female   22.0   UNKNOWN


In [3]:
# Identify and drop duplicate records
duplicates_found = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

print(f"Removed {duplicates_found} duplicate rows. Remaining rows: {len(df)}")

Removed 0 duplicate rows. Remaining rows: 100


In [6]:
# 1. Standardize Gender formatting
gender_map = {
    'Male': 'Male', 'male': 'Male', 'M': 'Male',
    'Female': 'Female', 'female': 'Female', 'F': 'Female'
}
df['Gender'] = df['Gender'].map(gender_map) # Unmapped NaN stays NaN

# 2. Convert Join_Date to Datetime (using format='mixed' to avoid dayfirst warnings)
df['Join_Date'] = pd.to_datetime(df['Join_Date'], errors='coerce', format='mixed')

# 3. Clean Income string to numeric Float
df['Income'] = df['Income'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['Income'] = pd.to_numeric(df['Income'], errors='coerce')

# 4. Standardize Customer_ID as String
df['Customer_ID'] = df['Customer_ID'].astype(str)

print("Updated Data Types:")
print(df.dtypes)

Updated Data Types:
Customer_ID               str
Join_Date      datetime64[us]
Gender                    str
Age                   float64
Income                float64
dtype: object


In [7]:
# Handle logical range anomalies in Age (-5, 150)
df.loc[(df['Age'] < 0) | (df['Age'] > 100), 'Age'] = np.nan

# Outlier Treatment for Income using IQR
Q1 = df['Income'].quantile(0.25)
Q3 = df['Income'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

# Cap extreme Income outliers at upper boundary
df['Income'] = np.where(df['Income'] > upper_bound, upper_bound, df['Income'])

print(df[['Age', 'Income']].describe().T)

        count          mean           std      min      25%      50%  \
Age      60.0     31.233333      7.990178     22.0     22.0     30.0   
Income   65.0  74077.061538  31842.986866  45000.0  45000.0  65000.5   

             75%       max  
Age         35.0      45.0  
Income  120000.0  120000.0  


In [8]:
# 1. Mode Imputation for Categorical (Gender)
gender_mode = df['Gender'].mode()[0]
df['Gender'] = df['Gender'].fillna(gender_mode)

# 2. Median Imputation for Skewed Numerical (Age & Income)
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Income'] = df['Income'].fillna(df['Income'].median())

# 3. Forward Fill for Sequential Dates (Join_Date)
df['Join_Date'] = df['Join_Date'].ffill().bfill()

print("Remaining Null Values:")
print(df.isnull().sum())

Remaining Null Values:
Customer_ID    0
Join_Date      0
Gender         0
Age            0
Income         0
dtype: int64


In [9]:
# Record clean state
after_rows = len(df)
after_nulls = df.isnull().sum().sum()
after_duplicates = df.duplicated().sum()

# Construct Comparison Table
summary_data = {
    'Metric': ['Total Rows', 'Null Values Count', 'Duplicate Rows', 'Data Types Accuracy'],
    'Before Cleaning': [before_rows, before_nulls, before_duplicates, 'Incorrect (Strings/Objects)'],
    'After Cleaning': [after_rows, after_nulls, after_duplicates, '100% Correct Dtypes']
}

summary_df = pd.DataFrame(summary_data)
print("=== BEFORE vs. AFTER CLEANING SUMMARY ===")
print(summary_df.to_string(index=False))

# Export clean dataset
df.to_csv('cleaned_retail_data.csv', index=False)
print("\nCleaned dataset exported to 'cleaned_retail_data.csv'.")

=== BEFORE vs. AFTER CLEANING SUMMARY ===
             Metric             Before Cleaning      After Cleaning
         Total Rows                         100                 100
  Null Values Count                          71                   0
     Duplicate Rows                           0                   1
Data Types Accuracy Incorrect (Strings/Objects) 100% Correct Dtypes

Cleaned dataset exported to 'cleaned_retail_data.csv'.
